<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/phonchi/CryoParticleSegment/blob/main/notebook/05_cryosparc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
</table>

## ⭐ Setup
You must run all codes under this category.

### ✅ Directory Settings

In [ ]:
# @title  { display-mode: "form" }
# @markdown If you had downloaded the data before and want to try different Denoise methods, choose the denoise methods so the code can locate the micrograph correctly for each denoised dataset.

denoised = "Topaz Denoise" # @param ["Raw", "Topaz Denoise", "Conv CryoSegNet Denoise Process"]

denoise_prefix = ""
if denoised == "Topaz Denoise":
    denoise_prefix = "TpzD_"
if denoised == "Conv CryoSegNet Denoise Process":
    denoise_prefix = "C-CSN-D_"

In [ ]:
# @title  { display-mode: "form" }
# @markdown Directory parameters.
import os

DATA_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/data" # @param {type:"string"}
RESULT_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/TpzD_output" # @param {type:"string"}
STAR_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/results/10017/unet_eb5_dice_CDCRF" # @param {type:"string"}
TMP_DIR = "/content/tmp" # @param {type:"string"}
SOFTWARE_DIR = "/content/software" # @param {type:"string"}
WORK_DIR = os.getcwd()

In [ ]:
# @title  { display-mode: "form" }
# @markdown Run this block if using folder📁 in Google Drive as **`RESULT_DIR`**.

try:
  from google.colab import drive
  drive.mount('/content/drive')
except:
  pass

Mounted at /content/drive


In [ ]:
# @title  { display-mode: "form" }
# @markdown Run this block for remove the **`sample_data`** folder📁 in content

if os.path.isdir("/content/sample_data"):
  !rm -r /content/sample_data
# from shutil import rmtree
#
# rmtree(f"/content/sample_data")

In [ ]:
# @title  { display-mode: "form" }
# @markdown Run this block for checking the existence of the directories

if not os.path.isdir(DATA_DIR):
  os.mkdir(DATA_DIR)

if not os.path.isdir(RESULT_DIR):
  os.mkdir(RESULT_DIR)

if not os.path.isdir(TMP_DIR):
  os.mkdir(TMP_DIR)

if not os.path.isdir(SOFTWARE_DIR):
  os.mkdir(SOFTWARE_DIR)

### ✅ EMPIAR Data Setting

In [ ]:
# @title  { vertical-output: true, display-mode: "form" }
EMPIAR_ID = 10017 # @param {type:"integer"}

In [ ]:
EMPIAR_DIR = os.path.join(DATA_DIR, f"EMPIAR-{EMPIAR_ID}")
EMPIAR_ID, DATA_DIR, RESULT_DIR, EMPIAR_DIR

(10017,
 '/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/data',
 '/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/TpzD_output',
 '/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/data/EMPIAR-10017')

In [ ]:
!mkdir {EMPIAR_DIR}

mkdir: cannot create directory ‘/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/data/EMPIAR-10017’: File exists


In [ ]:
# @title Copy Data to `->/content/data`
!mkdir /content/data
!rsync -av {RESULT_DIR}/dataset/10017/ground_truth  /content/data
!rsync -av {RESULT_DIR}/dataset/10017/micrographs /content/data

!cp {STAR_DIR}/cv_particles.star /content/data
!cp {STAR_DIR}/tp_particles.star /content/data
!cp {STAR_DIR}/nms_particles.star /content/data

sending incremental file list
rsync: [sender] change_dir "/content/{RESULT_DIR}/dataset/10017" failed: No such file or directory (2)
created directory {EMPIAR_DIR}

sent 19 bytes  received 47 bytes  132.00 bytes/sec
total size is 0  speedup is 0.00
rsync error: some files/attrs were not transferred (see previous errors) (code 23) at main.c(1338) [sender=3.2.7]
sending incremental file list
rsync: [sender] change_dir "/content/{RESULT_DIR}/dataset/10017" failed: No such file or directory (2)

sent 19 bytes  received 12 bytes  62.00 bytes/sec
total size is 0  speedup is 0.00
rsync error: some files/attrs were not transferred (see previous errors) (code 23) at main.c(1338) [sender=3.2.7]
cp: cannot stat '{STAR_DIR}/cv_particles.txt': No such file or directory
cp: cannot stat '{STAR_DIR}/cv_particles.txt': No such file or directory


#### 🟪 CryoSPARC Installation

In [ ]:
# @title  { display-mode: "form" }

INSTALL = True # @param {type:"boolean"}

In [ ]:
# @title  { display-mode: "form" }
# @markdown Download and extract CryoSPARC.

if INSTALL:
  try:
    LICENSE_ID = '9657b740-93cb-11f0-a34d-cb5f4bdb076d' # @param {type:"string"}
  except:
    pass

  os.environ['LICENSE_ID'] = LICENSE_ID

  %cd {SOFTWARE_DIR}
  %mkdir cryosparc
  %cd cryosparc
  if not os.path.isdir("cryosparc_master"):
    if not os.path.exists("cryosparc_master.tar.gz"):
      !curl -L https://get.cryosparc.com/download/master-latest/$LICENSE_ID -o cryosparc_master.tar.gz
    !tar -xf cryosparc_master.tar.gz cryosparc_master
    !rm /content/software/cryosparc/cryosparc_master.tar.gz
  if not os.path.isdir("cryosparc_worker"):
    if not os.path.exists("cryosparc_worker.tar.gz"):
      !curl -L https://get.cryosparc.com/download/worker-latest/$LICENSE_ID -o cryosparc_worker.tar.gz
    !tar -xf cryosparc_worker.tar.gz cryosparc_worker
    !rm /content/software/cryosparc/cryosparc_worker.tar.gz

/content/software
/content/software/cryosparc
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:--  0:00:03 --:--:--     0
100  630M  100  630M    0     0  17.4M      0  0:00:36  0:00:36 --:--:-- 21.1M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 3993M  100 3993M    0     0  17.2M      0  0:03:51  0:03:51 --:--:-- 16.9M


The refinement process should be implemented in cryoSPARC as follows:

1. Import Micrographs.

2. CTF Estimation.

3. Import Particles (only coordinates).

4. Etract from Microgaphs.

5. Ab-initio Reconstruction.

6. Homogeneous Refinement.


In [ ]:
# @title { display-mode: "form" }
# @markdown ## Install CryoSPARC

INSTALL = True  # @param {type:"boolean"}
EMAIL = "yki.016@g-mail.nsysu.edu.tw"  # @param {type:"string"}
PASSWORD = "SINRONG812"  # @param {type:"string"}
USERNAME = "ANIMATH"  # @param {type:"string"}
FIRST_NAME = "Sin-Rong"  # @param {type:"string"}
LAST_NAME = "Tsai"  # @param {type:"string"}
PORT = 61000  # @param {type:"integer"}

import os

if INSTALL:
  %cd cryosparc_master
  !apt-get install iputils-ping -y
  %mkdir -p /content/software/cryosparc/cryosparc_cache

  os.environ['LICENSE_ID'] = LICENSE_ID
  os.environ['USER'] = USERNAME
  %env CRYOSPARC_FORCE_USER=true

  !./install.sh --standalone \
      --license $LICENSE_ID \
      --worker_path /content/software/cryosparc/cryosparc_worker \
      --ssdpath /content/software/cryosparc/cryosparc_cache \
      --initial_email "{EMAIL}" \
      --initial_password "{PASSWORD}" \
      --initial_username "{USERNAME}" \
      --initial_firstname "{FIRST_NAME}" \
      --initial_lastname "{LAST_NAME}" \
      --port {PORT} \
      --yes
  %cd {WORK_DIR}

/content/software/cryosparc/cryosparc_master
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  iputils-ping
0 upgraded, 1 newly installed, 0 to remove and 38 not upgraded.
Need to get 43.0 kB of archives.
After this operation, 116 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 iputils-ping amd64 3:20211215-1ubuntu0.1 [43.0 kB]
Fetched 43.0 kB in 1s (60.6 kB/s)
Selecting previously unselected package iputils-ping.
(Reading database ... 126675 files and directories currently installed.)
Preparing to unpack .../iputils-ping_3%3a20211215-1ubuntu0.1_amd64.deb ...
Unpacking iputils-ping (3:20211215-1ubuntu0.1) ...
Setting up iputils-ping (3:20211215-1ubuntu0.1) ...
Processing triggers for man-db (2.10.2-1) ...
env: CRYOSPARC_FORCE_USER=true

************ CRYOSPARC SYSTEM: STANDALONE INSTALLER **************

You are the root user. Are you s

In [ ]:
# @title  { display-mode: "form" }
# @markdown Install pyngrok.

if INSTALL:
  %pip install pyngrok -qq

#### 🟪 Pyem Installation

In [ ]:
# @title  { display-mode: "form" }

INSTALL = True # @param {type:"boolean"}

if INSTALL:
  %cd {SOFTWARE_DIR}
  !pip install pyFFTW
  !pip install healpy
  !pip install pathos
  !git clone https://github.com/asarnow/pyem.git
  %cd pyem
  !pip install --no-dependencies -e .
  %cd {WORK_DIR}

/content/software
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.3/82.3 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.3/150.3 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 7.8 MB/s eta 0:00:00
  Attempting uninstall: dill
    Found existing installation: dill 0.3.8
    Uninstalling dill-0.3.8:
      Successfully uninstalled dill-0.3.8
  Attempting uninstall: multiprocess
    Found existing installation: multiprocess 0.70.16
    Uninstalling multiprocess-0.70.16:
      Successfully uninstalled multiprocess-0.70.16
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.

# Run CryoSparc

In [ ]:
from pyngrok import ngrok, conf
import getpass

In [ ]:
print("Enter your authtoken, which can be copied from https://dashboard.ngrok.com/")
conf.get_default().auth_token = getpass.getpass()

Enter your authtoken, which can be copied from https://dashboard.ngrok.com/
··········


In [ ]:
# Setup a tunnel to the port 61000
public_url = ngrok.connect(61000)

In [ ]:
public_url

<NgrokTunnel: "https://20994c1cc5cc.ngrok.app" -> "http://localhost:61000">

Replace `CS-new` with the name of your project to create a backup.


In [ ]:
RESULT_DIR

'/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/TpzD_output'

In [ ]:
!cd /content

In [ ]:
!rsync -av CS-new {RESULT_DIR}

To remain active in colab, execute the following code:

In [ ]:
import time

# Time to sleep in seconds (2 hours)
hours_to_sleep = 4
seconds_per_hour = 3600

# Calculate total sleep time in seconds
total_sleep_time = hours_to_sleep * seconds_per_hour

print("Going to sleep for", hours_to_sleep, "hour(s).")
time.sleep(total_sleep_time)
print("Woke up!")

Going to sleep for 4 hour(s).


KeyboardInterrupt: 